## Imports

In [1]:
import os
import sys

REPO_ROOT = os.path.abspath('..')
sys.path.insert(0, REPO_ROOT)                              # for scripts.evaluation
sys.path.insert(0, os.path.join(REPO_ROOT, 'scripts'))    # for pipeline's internal imports (vlm, rag, ocr)

import nest_asyncio
nest_asyncio.apply()

import pandas as pd
import json
from tqdm.notebook import tqdm
from scripts.pipeline import MMRAGPipeline
from scripts.evaluation import evaluate_dataframe

## Configurations

In [2]:
SEED = 19
QUESTION_PATHS = {
    "FinancialReport" : "../data/raw/REAL-MM-RAG_FinReport/test/qas.jsonl",
    "TechReport" : "../data/raw/REAL-MM-RAG_TechReport/test/qas.jsonl",
    "TechSlides" : "../data/raw/REAL-MM-RAG_TechSlides/test/qas.jsonl"
}
RESULTS_PATH = "./results/image_rag"
# Set model_id and checkpoint_path to match what is available locally.
# EC2 (A100/H100): use "Qwen/Qwen3-VL-8B-Instruct" / "../Qwen3-VL-8B-Instruct"
# Mac (local):     use "Qwen/Qwen3-VL-2B-Instruct" / "../Qwen3-VL-2B-Instruct"
model_id = "Qwen/Qwen3-VL-2B-Instruct"
checkpoint_path = "../Qwen3-VL-2B-Instruct"
EMBEDDINGS_DIR = "../scripts/embeddings"
TOP_K = 5

## Load Data

In [3]:
dataframes = {}

for key, path in QUESTION_PATHS.items():
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            records.append({
                "example_index": obj.get("example_index"),
                "question": obj.get("question"),
                "answer": obj.get("answer"),
                "intended_img": obj.get("image_path")
            })
    dataframes[key] = pd.DataFrame(records).set_index("example_index")

splits = {}
for key, df in dataframes.items():
    train = df.sample(frac=0.8, random_state=SEED)
    test = df.drop(train.index)
    splits[key] = {"train": train, "test": test}
    print(f"{key}: {len(train)} train / {len(test)} test")

FinancialReport: 8 train / 2 test
TechReport: 8 train / 2 test
TechSlides: 8 train / 2 test


## Initialize Pipeline

In [4]:
pipeline = MMRAGPipeline(
    mode="image_rag",
    vlm_model_id=model_id,
    vlm_checkpoint_path=checkpoint_path,
    image_retriever_persist_dir=EMBEDDINGS_DIR,
    top_k=TOP_K,
)

Initializing MMRAGPipeline | mode=image_rag | device=mps
--- Local model found at: ../Qwen3-VL-2B-Instruct ---
  Import error: No module named 'triton'


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/725 [00:00<?, ?it/s]

## Ingest Datasets

Changed this to a QdrantDB instead. Better for late interaction retrieval and image embeddings.

In [5]:
for name, qas_path in QUESTION_PATHS.items():
    print(f"Ingesting {name}...")
    pipeline.ingest(qas_path)

print("\n✅ All datasets ingested.")

Ingesting FinancialReport...


Indexing Pages: 100%|██████████| 10/10 [00:17<00:00,  1.80s/it]


Ingesting TechReport...


Indexing Pages: 100%|██████████| 10/10 [00:17<00:00,  1.77s/it]


Ingesting TechSlides...


Indexing Pages: 100%|██████████| 10/10 [00:19<00:00,  1.93s/it]


✅ All datasets ingested.


## Inference

In [6]:
os.makedirs(RESULTS_PATH, exist_ok=True)
output_filepath = f"{RESULTS_PATH}/image_rag_results.jsonl"

with open(output_filepath, "w", encoding="utf-8") as f:

    for key, split in splits.items():
        test_df = split['test']
        print(f"Running Inference for {key}")

        for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
            try:
                generated = pipeline.generate(row['question'])
            except Exception as e:
                print(f"Error on example {idx}: {e}")
                continue

            result_obj = {
                "dataset": key,
                "example_index": idx,
                "question": row['question'],
                "generated_answer": generated,
                "ground_truth": row['answer'],
                "intended_img": row['intended_img']
            }

            f.write(json.dumps(result_obj) + "\n")
            f.flush()

print(f"✅ Inference complete. Results saved continuously to {output_filepath}")

Running Inference for FinancialReport


  0%|          | 0/2 [00:00<?, ?it/s]

Keyword argument `return_dict` is not a valid argument for this processor and will be ignored.


Running Inference for TechReport


  0%|          | 0/2 [00:00<?, ?it/s]

Running Inference for TechSlides


  0%|          | 0/2 [00:00<?, ?it/s]

✅ Inference complete. Results saved continuously to ./results/image_rag/image_rag_results.jsonl


## Evaluation

In [7]:
input_filepath = f"{RESULTS_PATH}/image_rag_results.jsonl"
report_filepath = f"{RESULTS_PATH}/image_rag_evaluation_report.txt"
detailed_jsonl_filepath = f"{RESULTS_PATH}/image_rag_evaluated_details.jsonl"

print(f"Loading data from {input_filepath}...")
records = []
if os.path.exists(input_filepath):
    with open(input_filepath, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
else:
    raise FileNotFoundError(f"Could not find {input_filepath}. Check your paths!")

df_results = pd.DataFrame(records)
print(f"Loaded {len(df_results)} records across datasets: {df_results['dataset'].unique()}")
print("\nStarting evaluation (this may take a few minutes depending on BERTScore)...")
evaluated_df = evaluate_dataframe(df_results, report_filepath)

evaluated_df.to_json(detailed_jsonl_filepath, orient="records", lines=True)
print(f"✅ Detailed row-by-row scores saved to {detailed_jsonl_filepath}")

Loading data from ./results/image_rag/image_rag_results.jsonl...
Loaded 6 records across datasets: ['FinancialReport' 'TechReport' 'TechSlides']

Starting evaluation (this may take a few minutes depending on BERTScore)...
Calculating BLEU scores...
Calculating ROUGE scores...
Calculating BERT scores (this may take a moment)...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Report successfully saved to ./results/image_rag/image_rag_evaluation_report.txt
✅ Detailed row-by-row scores saved to ./results/image_rag/image_rag_evaluated_details.jsonl
